In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Robustness Test (Noise Injection)

In [ ]:
import random
import string
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Function to add noise to text
def add_noise(text, noise_level=0.1, noise_type='typo'):
    """
    noise_level: fraction of characters/words to modify (0 to 1)
    noise_type: 'typo', 'punctuation', 'case', 'word_drop'
    """
    text = str(text)
    words = text.split()
    
    if noise_type == 'typo':
        # Random character substitution
        chars = list(text)
        if len(chars) == 0:
            return text
        num_changes = max(1, int(len(chars) * noise_level))
        for _ in range(num_changes):
            idx = random.randint(0, len(chars)-1)
            chars[idx] = random.choice(string.ascii_lowercase)
        return ''.join(chars)
    
    elif noise_type == 'punctuation':
        # Remove or add punctuation
        chars = list(text)
        if len(chars) == 0:
            return text
        num_changes = max(1, int(len(chars) * noise_level))
        for _ in range(num_changes):
            idx = random.randint(0, len(chars)-1)
            if chars[idx] in '!?.,;:':
                chars[idx] = ''
            else:
                chars[idx] = random.choice('!?.,;:')
        return ''.join(chars)
    
    elif noise_type == 'case':
        # Randomly change case
        chars = list(text)
        if len(chars) == 0:
            return text
        num_changes = max(1, int(len(chars) * noise_level))
        for _ in range(num_changes):
            idx = random.randint(0, len(chars)-1)
            chars[idx] = chars[idx].swapcase()
        return ''.join(chars)
    
    elif noise_type == 'word_drop':
        # Randomly drop words
        if len(words) <= 2:
            return text
        num_drop = max(1, int(len(words) * noise_level))
        keep_indices = sorted(random.sample(range(len(words)), len(words) - num_drop))
        return ' '.join([words[i] for i in keep_indices])
    
    return text

In [ ]:
# Load Kaggle test data (original clean texts)
_, X_test_kaggle, _, y_test_kaggle = joblib.load('/kaggle/working/data_splits.pkl')

# Take a sample for faster testing (e.g., 500 samples)
np.random.seed(42)
sample_size = min(500, len(y_test_kaggle))
sample_idx = np.random.choice(len(y_test_kaggle), sample_size, replace=False)

X_sample = X_test_kaggle.iloc[sample_idx].values
y_sample = y_test_kaggle.iloc[sample_idx].values

print(f"Testing on {sample_size} samples from Kaggle test set")

# Define noise levels and types
noise_levels = [0, 0.05, 0.10, 0.15, 0.20]
noise_types = ['typo', 'punctuation', 'case', 'word_drop']
results = {nt: [] for nt in noise_types}

In [ ]:
# Run robustness test (using your stacking ensemble)
print("\nRunning robustness test...\n")

for noise_type in noise_types:
    for level in noise_levels:
        # Apply noise to texts
        noisy_texts = [add_noise(t, noise_level=level, noise_type=noise_type) for t in X_sample]
        
        # Clean noisy texts (same cleaning as training)
        noisy_clean = [clean_text(t) for t in noisy_texts]
        
        # Extract features using saved vectorizers
        noisy_word = word_vec.transform(noisy_clean)
        noisy_char = char_vec.transform(noisy_clean)
        
        # Meta features for noisy texts
        noisy_meta_raw = get_meta(noisy_clean)
        noisy_meta_scaled = scaler.transform(noisy_meta_raw)
        noisy_meta = csr_matrix(noisy_meta_scaled)
        
        # Combine features
        noisy_combined = hstack([noisy_word, noisy_char, noisy_meta])
        
        # Predict using stacking ensemble
        p_lr_noisy = lr.predict_proba(noisy_combined)[:, 1].reshape(-1, 1)
        p_xgb_noisy = xgb.predict_proba(noisy_combined)[:, 1].reshape(-1, 1)
        p_lgbm_noisy = lgbm.predict_proba(noisy_combined)[:, 1].reshape(-1, 1)
        meta_input_noisy = np.hstack([p_lr_noisy, p_xgb_noisy, p_lgbm_noisy])
        y_pred_noisy = meta_model.predict(meta_input_noisy)
        
        acc = accuracy_score(y_sample, y_pred_noisy)
        results[noise_type].append(acc)
        print(f"{noise_type:12} | Noise: {int(level*100):3}% | Accuracy: {acc:.4f} ({acc*100:.2f}%)")

In [ ]:
# Plot robustness results
plt.figure(figsize=(10, 6))
for noise_type in noise_types:
    plt.plot([l*100 for l in noise_levels], [a*100 for a in results[noise_type]], 
             marker='o', label=noise_type.capitalize(), linewidth=2)
plt.xlabel('Noise Level (%)')
plt.ylabel('Accuracy (%)')
plt.title('Robustness Test: Accuracy vs Noise Level (Stacking Ensemble)')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(80, 100)
plt.savefig('/kaggle/working/robustness_test.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: robustness_test.png")

# Summary at 10% noise
print("\n" + "="*50)
print("ROBUSTNESS SUMMARY (at 10% noise)")
print("="*50)
for noise_type in noise_types:
    idx = noise_levels.index(0.10)
    acc_at_10 = results[noise_type][idx] * 100
    drop = 99.66 - acc_at_10
    print(f"{noise_type.capitalize()}: {acc_at_10:.2f}% (drop: {drop:.2f}%)")